# Load Jira Cloud issues into a Delta table

Full load on the first run, then incremental. Read-only against Jira Cloud.

**Before you run**
1. Upload `jira.py` to your workspace and set `HELPER_DIR` below to its folder.
2. Provide `JIRA_SITE` (`<site>.atlassian.net`), `JIRA_EMAIL` and `JIRA_API_TOKEN`
   (create one at https://id.atlassian.com/manage/api-tokens) as environment
   variables for the cluster. Do not type the token into a cell.
3. Set `TARGET` and `JQL` below.

In [ ]:
import sys
HELPER_DIR = "<WORKSPACE_PATH_TO_HELPER_FOLDER>"
TARGET = "<CATALOG>.<SCHEMA>.jira_issue"
JQL = "project = KAN"
PAGE_SIZE = 100
OVERLAP_SECONDS = 300
sys.path.insert(0, HELPER_DIR)

# Pin the Spark session to UTC so a TIMESTAMP read back with spark.sql(...)
# .first() comes back as a naive UTC datetime, matching what format_jql_timestamp
# assumes for a naive value. Without this, the session's local timezone would
# silently shift the incremental watermark.
spark.conf.set("spark.sql.session.timeZone", "UTC")

In [ ]:
from datetime import datetime, timezone
import jira as j

site, email, token = j.credentials_from_env()
session = j.jira_session(email, token)

# Jira interprets JQL date-time literals in the searching account's own
# timezone, not UTC. Fetch it once per run and pass it explicitly — omitting
# this silently skips or delays issues for any account not on UTC.
tz_name = j.account_timezone(session, site)

# Incremental: start from the newest row already loaded (None on the first run).
since = None
if spark.catalog.tableExists(TARGET):
    since = spark.sql(f"SELECT max(updated) AS m FROM {TARGET}").first()["m"]

until = datetime.now(timezone.utc)  # fixed for this run
issues = j.search_issues(
    session, site, query=JQL, tz_name=tz_name,
    since=since, until=until, overlap_seconds=OVERLAP_SECONDS, page_size=PAGE_SIZE,
)
df = j.to_dataframe(spark, list(issues))
print("issues read:", df.count())

In [ ]:
# The overlap window re-reads a few issues on purpose; MERGE on key removes duplicates.
if not spark.catalog.tableExists(TARGET):
    df.write.format("delta").saveAsTable(TARGET)
else:
    df.createOrReplaceTempView("incoming_issue")
    spark.sql(
        f"MERGE INTO {TARGET} t USING incoming_issue s ON t.key = s.key "
        "WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *"
    )
print("issues in target:", spark.table(TARGET).count())

## Limits
- An `updated` watermark does not see Jira issue deletes.
- An issue updated after `until` while the read is running is picked up by the
  next run.
- Custom fields (`customfield_NNNNN`) are not in the typed output; requesting
  one via `fields=` will not add a column (verified 2026-09-29, see
  `spike/RESULTS.md`).
- **Jira reads JQL date-time literals in the searching account's own timezone,
  not UTC** (verified 2026-09-29; observed `+0500`, not `+0000`). This notebook
  fetches that timezone once per run via `account_timezone()` and passes it as
  `tz_name=`; a caller of `search_issues` who omits `tz_name` gets UTC bounds,
  which silently skips or delays issues for any account not on UTC.
- This notebook pins `spark.sql.session.timeZone` to UTC so the watermark read
  back from Delta is unambiguous; without that, the Spark session's own
  timezone setting could shift `since`.
- A personal/free Jira Cloud site is for learning and testing, not production
  volume.